# 1. Conda 环境与首次构建 COF（Windows 版）

语言：[English](../en/01_environment_and_first_build.ipynb) | **中文**

本原生 Windows 版 Notebook 将创建仓库定义的 `cofkit` Conda 环境、把它注册为 Jupyter 内核，并在 `hcb` 拓扑上构建 TAPB–TFB 亚胺 COF。它使用普通 Python 单元格，不需要 PowerShell 或 WSL。请在本仓库的克隆目录中运行。

可执行单元格通过 Python 的 `subprocess` 模块传递 CLI 参数。每个主要操作下方都提供了完全注释掉的 Python 单元格，用于展示等价的 API 工作流，但不会执行。

## 教程系列

1. **环境与首次构建**（本 Notebook）
2. [拓扑、连接数组合与连接键示例](02_topologies_connectivities_and_linkages.ipynb)
3. [使用 Zeo++ 进行孔隙分析](03_zeopp_pore_analysis.ipynb)

请从仓库根目录启动 Jupyter，确保下文所有相对路径都能一致解析。Python 单元格通过 `subprocess` 使用 `conda run -n cofkit`，因此无需让内核的创建或激活状态在不同单元格之间保持。

## 查看仓库中的环境定义

`environment.yml` 的第一行将环境名称固定为 `cofkit`。整个教程系列都使用这个名称。

In [ ]:
from pathlib import Path

repo_root = Path.cwd().resolve()
environment_file = repo_root / "environment.yml"
if not environment_file.is_file():
    raise FileNotFoundError(
        "请从 cofkit 仓库根目录启动 Jupyter，然后重新运行此单元格。"
    )

print(f"仓库根目录：{repo_root}\n")
print("\n".join(environment_file.read_text(encoding="utf-8").splitlines()[:24]))

In [ ]:
# Python 等价实现（已注释）：查看环境名称。
# from pathlib import Path
# environment_name = next(
#     line.split(":", 1)[1].strip()
#     for line in Path("environment.yml").read_text().splitlines()
#     if line.startswith("name:")
# )
# print(environment_name)

## 创建或更新环境

这个单元格会先检查 `cofkit` CLI 是否已在 `PATH` 中可用。如果可用，单元格将成功退出且不修改环境，这在选择 **Run All**（全部运行）时尤其有用。否则，若 `cofkit` 环境不存在则创建它，存在则根据 `environment.yml` 更新；随后安装本地软件包而不替换 Conda 已解析的依赖，并为 Notebooks 2 和 3 注册 `Python (cofkit)` 内核。

> 首次运行时，环境依赖解析和 Open Babel 安装可能需要几分钟。

In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess

cofkit_command = shutil.which("cofkit")
if cofkit_command:
    print(
        f"cofkit CLI 已可用，位置：{cofkit_command}；"
        "跳过环境和软件包设置。"
    )
else:
    repo_root = Path.cwd().resolve()
    environment_file = repo_root / "environment.yml"
    if not environment_file.is_file():
        raise FileNotFoundError(
            "请从 cofkit 仓库根目录启动 Jupyter，然后重新运行此单元格。"
        )

    conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
    if not conda:
        raise RuntimeError(
            "未找到 Conda。请从 Miniforge、Miniconda 或 Anaconda "
            "启动 Jupyter，然后重新运行此单元格。"
        )

    environment_info = json.loads(
        subprocess.run(
            [conda, "env", "list", "--json"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout
    )
    environment_exists = any(
        Path(path).name.casefold() == "cofkit"
        for path in environment_info["envs"]
    )
    if environment_exists:
        environment_command = [
            conda, "env", "update", "--name", "cofkit",
            "--file", str(environment_file), "--prune",
        ]
    else:
        environment_command = [
            conda, "env", "create", "--file", str(environment_file),
        ]
    subprocess.run(environment_command, cwd=repo_root, check=True)
    subprocess.run(
        [conda, "run", "-n", "cofkit", "python", "-m", "pip",
         "install", "--no-deps", "."],
        cwd=repo_root,
        check=True,
    )
    subprocess.run(
        [conda, "install", "--name", "cofkit", "--channel", "conda-forge",
         "--yes", "ipykernel"],
        cwd=repo_root,
        check=True,
    )
    subprocess.run(
        [conda, "run", "-n", "cofkit", "python", "-m", "ipykernel",
         "install", "--user", "--name", "cofkit",
         "--display-name", "Python (cofkit)"],
        cwd=repo_root,
        check=True,
    )
    print("cofkit 环境和 Jupyter 内核已准备完成。")

In [ ]:
# Conda 没有用于创建环境的稳定公共 Python API。
# 因此，上面的可运行单元格通过 subprocess 参数安全地调用 Conda 和 pip。

如果本 Notebook 是用其他内核打开的，现在可以从 Jupyter 的内核菜单中选择 **Python (cofkit)**。后续可执行单元格都会显式使用 `conda run -n cofkit`，因此在这里切换内核并非必需。Notebooks 2 和 3 已将 `cofkit` 声明为默认内核。

## 验证安装

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("未找到 Conda。请先运行环境设置单元格。")

commands = [
    [conda, "run", "-n", "cofkit", "cofkit", "--version"],
    [conda, "run", "-n", "cofkit", "python", "-c",
     "import cofkit; print('Python import:', cofkit.__version__)"],
    [conda, "run", "-n", "cofkit", "cofkit", "build", "list-templates"],
]
for command in commands:
    subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python 等价实现（已注释）：验证软件包能够导入。
# import cofkit
# print(cofkit.__version__)
#
# from cofkit import ReactionLibrary
# library = ReactionLibrary.builtin()
# print(library)

## 在 `hcb` 拓扑上构建 TAPB–TFB

TAPB 和 TFB 各自具有三个反应基团，因此形成 3+3 连接数组合。`imine_bridge` 模板连接胺基和醛基，而 `--topology hcb` 则让拓扑选择明确且可复现。

该命令会写入 `summary.json`，并把导出的 CIF 按粗略验证类别存放在 `out/tutorial_01_first_build/cifs/` 下。

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

repo_root = Path.cwd().resolve()
conda = os.environ.get("CONDA_EXE") or shutil.which("conda")
if not conda:
    raise RuntimeError("未找到 Conda。请先运行环境设置单元格。")

command = [
    conda, "run", "-n", "cofkit", "cofkit",
    "build", "single-pair",
    "--template-id", "imine_bridge",
    "--first-smiles", "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
    "--second-smiles", "C1=C(C=C(C=C1C=O)C=O)C=O",
    "--first-id", "tapb",
    "--second-id", "tfb",
    "--first-motif-kind", "amine",
    "--second-motif-kind", "aldehyde",
    "--topology", "hcb",
    "--target-dimensionality", "2D",
    "--output-dir", "out/tutorial_01_first_build",
]
subprocess.run(command, cwd=repo_root, check=True)

In [ ]:
# Python API 等价实现（已注释）。
# from pathlib import Path
# from cofkit import BatchGenerationConfig, BatchStructureGenerator, build_rdkit_monomer
#
# tapb = build_rdkit_monomer(
#     "tapb",
#     "TAPB",
#     "C1=CC(=CC=C1C2=CC(=CC(=C2)C3=CC=C(C=C3)N)C4=CC=C(C=C4)N)N",
#     "amine",
#     num_conformers=4,
# )
# tfb = build_rdkit_monomer(
#     "tfb",
#     "TFB",
#     "C1=C(C=C(C=C1C=O)C=O)C=O",
#     "aldehyde",
#     num_conformers=4,
# )
# generator = BatchStructureGenerator(
#     BatchGenerationConfig(
#         allowed_reactions=("imine_bridge",),
#         target_dimensionality="2D",
#         topology_ids=("hcb",),
#         rdkit_num_conformers=4,
#         max_workers=1,
#         write_cif=True,
#     )
# )
# summaries, candidates, attempted = generator.generate_monomer_pair_candidates(
#     tapb,
#     tfb,
#     pair_id="tapb__tfb",
#     out_dir=Path("out/tutorial_01_first_build_python/cifs"),
# )
# print("attempted:", attempted)
# for summary in summaries:
#     print(summary.topology_id, summary.status, summary.score, summary.cif_path)

## 查看构建输出

CIF 可能出现在 `valid`、`warning` 或 `needs_optimization` 目录中；这些目录表示粗略的几何分类，而不是不同的化学体系。Notebook 3 会搜索所有这些子目录，不会预设某一个类别。

In [ ]:
import json
from pathlib import Path

output_dir = Path("out/tutorial_01_first_build")
summary_path = output_dir / "summary.json"
if not summary_path.is_file():
    raise FileNotFoundError("请先运行首次构建单元格，再查看输出。")

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print("--- summary.json ---")
print(json.dumps(summary, indent=2))
print("\n--- 导出的 CIF 文件 ---")
print(*(path.resolve() for path in sorted((output_dir / "cifs").rglob("*.cif"))), sep="\n")

In [ ]:
# Python 等价实现（已注释）：查看结构化报告并查找 CIF。
# import json
# from pathlib import Path
# output_dir = Path("out/tutorial_01_first_build")
# summary = json.loads((output_dir / "summary.json").read_text())
# print(summary["successful_structures"], summary["cifs_written"])
# print(*sorted((output_dir / "cifs").rglob("*.cif")), sep="\n")

## 需要保留和关注的内容

- `summary.json` 记录了所请求的化学体系、检测到的基元数量、尝试的结构、评分、验证标记和 CIF 路径。
- 每个 CIF 都以 `# COFid:` 注释开头，其中包含单体、拓扑和连接键。
- 这些是初始生成结构。标记为警告或 `needs_optimization` 的结果，应在模拟或发表前完成优化并重新验证。
- 请保留 `out/tutorial_01_first_build/`：Notebook 3 会把其中的 CIF 作为默认 Zeo++ 输入。

继续学习 [Notebook 2](02_topologies_connectivities_and_linkages.ipynb)，比较少量具有代表性的其他构建选择。